# Appendix: additional experiments from prior notebooks and related literature

This notebook keeps experiments that are useful for appendix material: Zhao/Kang--Schafer balance paths, kernel-GP basis mismatch, ACIC, HDMA, and NSW randomized checks.  ATE and ATT are estimated whenever the corresponding wrapper is available.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings("once")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make the experiment helper importable whether Jupyter is launched from the
# repository root or from notebooks/experiments.
for _candidate in [Path.cwd(), Path.cwd() / "notebooks" / "experiments"]:
    if (_candidate / "grr_experiment_utils.py").exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

# Local experiment helpers. These live beside the notebooks and do not modify src/genriesz.
from grr_experiment_utils import *
from genriesz import (
    grr_ate, grr_att, ATEFunctional, ATTFunctional,
    BregmanGenerator, SquaredGenerator, UKLGenerator, BKLGenerator, BPGenerator,
    PolynomialBasis, TreatmentInteractionBasis,
)

# Optional random-forest leaf basis used in model-comparison experiments.
try:
    from sklearn.ensemble import RandomForestRegressor
    from genriesz.sklearn_basis import RandomForestLeafBasis
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not available; random-forest basis cells will fall back to RKHS.", exc)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

In [ ]:
FAST_MODE = True
POLYNOMIAL_DEGREE = 1 if FAST_MODE else 2
DOWNLOAD_DATA = False if FAST_MODE else True
os.environ["GRR_ALLOW_REMOTE_DATA"] = "1" if DOWNLOAD_DATA else "0"

# Every analysis estimates both targets when the wrapper supports them.
# If one target fails for a particular method, the failure row is kept and the other target is displayed.
ESTIMANDS = ("ate", "att")

N_REPS = 1 if FAST_MODE else 200
N = 40 if FAST_MODE else 3000
FOLDS = 2 if FAST_MODE else 5
MAX_ITER = 25 if FAST_MODE else 500
N_FEATURES = 4 if FAST_MODE else 120

LOSS_GRID = [("SQ", None), ("UKL", None), ("BKL", None), ("BP", 0.5)]
LAMBDA_MAIN = 1e-2
ESTIMATORS_ALL = ("ra", "rw", "arw", "tmle")
print(display_mode_banner(FAST_MODE))
TABLE_TITLE_ZHAO = "Appendix Table A4. Zhao/Kang-Schafer balance path for ATE and ATT."
FIGURE_TITLE_ZHAO = "Appendix Figure A4. Balance path by estimand and loss"
TABLE_TITLE_KERNEL = "Appendix Table A5. Kernel-GP basis mismatch for ATE and ATT."
TABLE_TITLE_EMP = "Appendix Table A6. Additional semi-synthetic and real-data checks for ATE and ATT."
ACTIVE_LOSS_GRID = LOSS_GRID[:2] if FAST_MODE else LOSS_GRID
ACTIVE_MAX_ACTIVE = 2 if FAST_MODE else 8
ACTIVE_DATASETS = ["ACIC"] if FAST_MODE else ["ACIC", "HDMA", "NSW"]


In [ ]:
def dgp_factory(name, *, n, seed):
    """Three DGPs used throughout the main simulation study."""
    if name == "DGP1 nonlinear heterogeneous":
        return make_ate_data(n=n, d=8, kappa=1.0, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP2 weak overlap":
        return make_ate_data(n=n, d=8, kappa=2.5, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP3 Kang-Schafer misspecification":
        return make_kang_schafer_data(n=n, seed=seed, tau=1.0)
    raise ValueError(name)


def basis_for_model(model, *, seed=0, n_features=N_FEATURES, sigma=1.0):
    """Editable basis map used by the GRR experiments."""
    model = str(model).lower()
    if model in {"rkhs", "gaussian"}:
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"poly", "polynomial"}:
        return make_treatment_basis("poly", degree=POLYNOMIAL_DEGREE, n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rff", "fourier", "random_fourier"}:
        return make_treatment_basis("rff", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rf", "random_forest"}:
        if SKLEARN_AVAILABLE:
            rf = RandomForestRegressor(n_estimators=8 if FAST_MODE else 80, max_depth=3 if FAST_MODE else 5,
                                       min_samples_leaf=5, random_state=seed)
            return TreatmentInteractionBasis(base_basis=RandomForestLeafBasis(rf, include_bias=True, normalize=True))
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    raise ValueError(model)


def loss_label(loss, omega=None):
    return loss if omega is None else f"{loss}({omega:g})"


def add_metadata(df, **kwargs):
    out = df.copy()
    for k, v in kwargs.items():
        out[k] = v
    return out


def clean_results(df):
    if "status" in df.columns:
        status = df["status"].fillna("ok")
    else:
        status = pd.Series("ok", index=df.index)
    return df[status.eq("ok") & df["estimator"].ne("failed")].copy()


def mc_table(df, group_cols, estimator_filter=None):
    d = clean_results(df)
    if estimator_filter is not None:
        d = d[d["estimator"].isin(list(estimator_filter))]
    return summarize_mc(d, group_cols)


def boxplot_metric(df, *, group_col, metric, title, estimator="arw", rotate=45, ylim=None):
    d = clean_results(df)
    if estimator is not None:
        d = d[d["estimator"] == estimator]
    labels = list(d[group_col].dropna().astype(str).unique())
    data = [d.loc[d[group_col].astype(str) == lab, metric].dropna().to_numpy() for lab in labels]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(labels)), 4.5))
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=rotate)
    if ylim is not None:
        ax.set_ylim(*ylim)
    fig.tight_layout()
    plt.show()
    return fig, ax


def run_one(data, *, estimand, loss, omega, basis, lam, cross_fit, folds, estimators=ESTIMATORS_ALL,
            penalty="l2", max_iter=MAX_ITER):
    """Fit ATE or ATT. Failures are returned as rows so tables remain complete."""
    try:
        df = fit_grr_estimand(
            data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=lam,
            cross_fit=cross_fit, folds=folds, estimators=estimators, penalty=penalty,
            max_iter=max_iter,
        )
        df["status"] = "ok"
        return df
    except Exception as exc:
        theta = true_theta_for_estimand(data, estimand)
        row = {
            "estimand": estimand.upper(), "estimator": "failed", "status": type(exc).__name__,
            "message": str(exc)[:240], "true_theta": theta,
        }
        return pd.DataFrame([row])

## A. Zhao/Kang--Schafer balance path

This reproduces the spirit of Zhao's tailored-loss toy example: features enter progressively, and standardized mean differences are displayed as a path.  Both ATE and ATT estimates are attempted.  The balance diagnostic uses the fitted Riesz weights for the requested target.

In [ ]:
def zhao_path_fit(data, *, estimand, loss, omega, n_active, lam=1e-2):
    base = SelectedColumnsBasis(columns=list(range(n_active)), include_bias=True)
    basis = ZOnlyBasis(base, treatment_index=0)
    gen = make_generator(loss, omega=omega)
    wrapper = grr_ate if estimand == "ate" else grr_att
    res = wrapper(X=data["X"], Y=data["Y"], basis=basis, generator=gen,
                  cross_fit=False, riesz_lam=lam, estimators=("rw",), max_iter=MAX_ITER, outcome_models="none")
    functional = ATEFunctional(treatment_index=0) if estimand == "ate" else ATTFunctional(treatment_index=0, pi=float(np.mean(data["D"])))
    alpha, _ = fit_grr_glm_alpha(data["X"], functional, basis, gen, lam=lam, max_iter=MAX_ITER)
    smd = smd_matrix(data["X"][:, 1:1+n_active], data["D"], weights=np.abs(alpha))
    df = result_to_frame(res, true_theta=true_theta_for_estimand(data, estimand))
    df["estimand"] = estimand.upper(); df["status"] = "ok"
    return float(smd["abs_smd"].max()), df

rows = []
path_rows = []
for rep in range(N_REPS):
    data = make_kang_schafer_data(n=N, seed=7100 + rep, tau=1.0)
    max_active = ACTIVE_MAX_ACTIVE
    for estimand in ESTIMANDS:
        for n_active in range(1, max_active + 1):
            for loss, omega in ACTIVE_LOSS_GRID:
                try:
                    max_smd, df = zhao_path_fit(data, estimand=estimand, loss=loss, omega=omega, n_active=n_active, lam=1e-2)
                    df = add_metadata(df, rep=rep, n_active=n_active, loss=loss_label(loss, omega), max_abs_smd=max_smd)
                    rows.append(df)
                    path_rows.append({"estimand": estimand.upper(), "rep": rep, "n_active": n_active, "loss": loss_label(loss, omega), "max_abs_smd": max_smd})
                except Exception as exc:
                    rows.append(pd.DataFrame([{"estimand": estimand.upper(), "rep": rep, "n_active": n_active, "loss": loss_label(loss, omega),
                                               "estimator": "failed", "status": type(exc).__name__, "message": str(exc)[:240]}]))

zhao_results = pd.concat(rows, ignore_index=True)
zhao_path = pd.DataFrame(path_rows)
print(TABLE_TITLE_ZHAO)
display(safe_display_frame(zhao_results.groupby(["estimand", "loss", "n_active", "estimator"], dropna=False).mean(numeric_only=True).reset_index(), n=120))
ACTIVE_LOSS_GRID = LOSS_GRID[:2] if FAST_MODE else LOSS_GRID
ACTIVE_MAX_ACTIVE = 2 if FAST_MODE else 8
ACTIVE_DATASETS = ["ACIC"] if FAST_MODE else ["ACIC", "HDMA", "NSW"]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, estimand in zip(axes, ["ATE", "ATT"]):
    dd = zhao_path[zhao_path["estimand"] == estimand]
    for loss, g in dd.groupby("loss"):
        gg = g.groupby("n_active")["max_abs_smd"].mean().reset_index()
        ax.plot(gg["n_active"], gg["max_abs_smd"], marker="o", label=loss)
    ax.axhline(0.10, linestyle="--", linewidth=1)
    ax.set_xlabel("Number of active transformed covariates")
    ax.set_ylabel("Maximum absolute SMD")
    ax.set_title(estimand)
    ax.legend(loc="best")
fig.suptitle(FIGURE_TITLE_ZHAO)
fig.tight_layout()
plt.show()

## B. Kernel-GP basis mismatch

Outcome and treatment assignment functions are generated from different kernels.  The experiment compares whether the fitted basis class can control ATE and ATT error under basis mismatch.

In [ ]:
rows = []
BASIS_GRID_KERNEL = ["rkhs", "rff", "polynomial"]
for rep in range(N_REPS):
    data = make_kernel_gp_data(n=N, d=5, f_kernel="poly1", g_kernel="gaussian", g_param=0.1, seed=8100 + rep)
    for estimand in ESTIMANDS:
        for model_name in BASIS_GRID_KERNEL:
            for loss, omega in ACTIVE_LOSS_GRID:
                basis = basis_for_model(model_name, seed=rep, n_features=N_FEATURES, sigma=1.0)
                df = run_one(data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=1e-2,
                             cross_fit=True, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER)
                df = add_metadata(df, rep=rep, model=model_name, loss=loss_label(loss, omega))
                rows.append(df)

kernel_results = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_KERNEL)
kernel_table = mc_table(kernel_results, ["estimand", "model", "loss", "estimator"])
display(safe_display_frame(kernel_table, n=120))

In [ ]:
plot_df = clean_results(kernel_results).query("estimator == 'arw'").copy()
plot_df["method"] = plot_df["estimand"] + " | " + plot_df["model"] + " | " + plot_df["loss"]
boxplot_metric(plot_df, group_col="method", metric="squared_error", title="Kernel-GP basis mismatch", estimator=None, rotate=70)

## C. ACIC, HDMA, and NSW randomized checks

ACIC is semi-synthetic and has target values for ATE and ATT when potential-outcome means are available.  HDMA is a real-data sensitivity analysis with no known ground truth.  NSW randomized is included as a stability check; ATT benchmark error is displayed when available.

In [ ]:
rows = []
# ACIC semi-synthetic
acic_conditions = [1] if FAST_MODE else [1, 2, 3, 4, 5]
for condition in acic_conditions:
    data = load_acic(condition=condition, fallback_seed=9000 + condition)
    data = subsample_data(data, n=400 if FAST_MODE else len(data["X"]), seed=condition)
    df = fit_ate_att_dataset(data, estimands=ESTIMANDS, loss_grid=ACTIVE_LOSS_GRID, lambdas=[1e-2], basis_kind="rkhs",
                             cross_fit=True, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER)
    df["dataset"] = f"ACIC condition {condition}"
    rows.append(df)

# HDMA real-data sensitivity: no true effect, so estimates and diagnostics only.
if "HDMA" in ACTIVE_DATASETS:
    hdma = load_hdma(fallback_seed=9100)
    hdma = subsample_data(hdma, n=500 if FAST_MODE else len(hdma["X"]), seed=1)
    df_hdma = fit_ate_att_dataset(hdma, estimands=ESTIMANDS, loss_grid=ACTIVE_LOSS_GRID, lambdas=[1e-2], basis_kind="rkhs",
                                  cross_fit=True, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER)
    df_hdma["dataset"] = "Boston HDMA"
    rows.append(df_hdma)

# NSW randomized-only stability check.
if "NSW" in ACTIVE_DATASETS:
    nsw = load_nsw_randomized(fallback_seed=9200)
    df_nsw = fit_ate_att_dataset(nsw, estimands=ESTIMANDS, loss_grid=ACTIVE_LOSS_GRID, lambdas=[1e-2], basis_kind="rkhs",
                                 cross_fit=True, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER)
    df_nsw["dataset"] = "NSW randomized"
    rows.append(df_nsw)

additional_results = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_EMP)
cols = ["dataset", "source", "estimand", "loss", "estimator", "estimate", "se", "bias", "squared_error", "benchmark", "benchmark_error", "alpha_abs_p95", "max_abs_smd_weighted"]
cols = [c for c in cols if c in additional_results.columns]
display(safe_display_frame(clean_results(additional_results)[cols], n=150))

In [ ]:
plot_df = clean_results(additional_results).copy()
plot_df = plot_df[plot_df["estimator"] == "arw"]
plot_df["method"] = plot_df["dataset"] + " | " + plot_df["estimand"] + " | " + plot_df["loss"]
fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(plot_df)), 4.5))
ax.scatter(plot_df["method"], plot_df["estimate"], s=50)
ax.set_title("Additional checks: ARW estimates")
ax.set_xlabel("Dataset | estimand | loss")
ax.set_ylabel("Estimate")
ax.tick_params(axis="x", rotation=80)
fig.tight_layout()
plt.show()